In [ ]:
# Google Colab Notebook for Implementing SDEE Algorithms

## Steps:
# 1. Load the data from 'sdee_lite.sql'
# 2. Clean the data (handle missing values, format conversion, etc.)
# 3. Implement DevSDEE, LOC Straw Man, ATLM, and ABE (k-NN)

# Install required libraries
!pip install pandas numpy scikit-learn sqlalchemy

import pandas as pd
import numpy as np
import sqlite3
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Connect to SQLite database and load data
conn = sqlite3.connect('/content/sdee_lite.sql')  # Update path as needed
query = "SELECT * FROM sdee_lite;"
df = pd.read_sql(query, conn)
conn.close()

# Display first few rows
df.head()

# Data Cleaning: Handle missing values
df.fillna(0, inplace=True)  # Replace NaN with 0

# Ensure numerical columns are in correct type
numeric_cols = ['dev_count', 'development_time', 'commit_count', 'loc_modified', 'effort_estimate']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Implementing DevSDEE
def calculate_dev_sdee(df):
    df['dev_sdee_effort'] = df['dev_count'] * df['development_time']
    return df

df = calculate_dev_sdee(df)

# Implementing LOC Straw Man
def calculate_loc_effort(df):
    df['loc_effort'] = df['loc_modified'] * 0.1  # Assuming 1 LOC = 0.1 person-hours
    return df

df = calculate_loc_effort(df)

# Implementing ATLM (Linear Regression)
X = df[['dev_count', 'loc_modified', 'development_time', 'commit_count']]
y = df['effort_estimate']

model = LinearRegression()
model.fit(X, y)
df['atlm_effort'] = model.predict(X)

# Implementing ABE (Analogy-Based Estimation using k-NN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(X_scaled)
distances, indices = knn.kneighbors(X_scaled)

# Compute weighted effort estimate
weights = 1 / (distances + 0.001)  # Avoid division by zero
df['abe_effort'] = np.sum(y.iloc[indices] * weights, axis=1) / np.sum(weights, axis=1)

# Display results
df[['repo_id', 'dev_sdee_effort', 'loc_effort', 'atlm_effort', 'abe_effort']].head()

# Save cleaned data with estimations
output_path = "/content/sdee_results.csv"
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
